In [1]:
# Chap 16. 다양한 댓글 모으기
# 뉴스 기사의 댓글 모으기 - 미세먼지 / 스모그
# 테스트 기사 URL :
#https://news.naver.com/main/read.nhn?mode=LSD&mid=shm&sid1=102&oid=056&
#aid=0010661268

#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import numpy
import pandas as pd
import random
import os
import re



In [2]:
#Step 2. 사용자에게 검색어 키워드를 입력 받고 저장할 폴더와 파일명을 설정합니다.
print("=" *80)
print(" Chap 16.뉴스 기사의 댓글 정보 수집하기")
print("=" *80)
print("\n")

query_txt = '뉴스기사댓글'
query_url = input('1.댓글을 크롤링할 뉴스의 URL을 입력하세요: ')
if query_url =='':
    query_url ='https://news.naver.com/main/read.nhn?mode=LSD&mid=shm&sid1=102&oid=056&aid=0010661268'
cnt = int(input('2.크롤링 할 건수는 몇건입니까?(10건단위로 입력): '))
page_cnt = math.ceil(cnt / 20)

f_dir = input("3.파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\):")
if f_dir=='' :
    f_dir='c:\\py_temp\\'

# 저장될 파일위치와 이름을 지정합니다
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s+'-'+query_txt)
os.chdir(f_dir+s+'-'+query_txt)

ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xlsx'



 Chap 16.뉴스 기사의 댓글 정보 수집하기




In [3]:
#Step 3. 크롬 드라이버를 사용해서 웹 브라우저를 실행합니다.

s_time = time.time( )

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get(query_url)
driver.maximize_window()
time.sleep(5)



In [4]:
#Step 4. 현재 총 리뷰 건수를 확인하여 사용자의 요청건수와 비교 후 동기화합니다
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 댓글 총 개수 써있는 영역 
result= soup.find('div', class_='u_cbox_head').find('span','u_cbox_count')
result2 = result.get_text() # 태그 안에 텍스트만 꺼냄 

# 실제 댓글 총 개수를 숫자로 만들기 
print("=" *80)
result3 = result2.replace(",","") # , 쉼표 없애기 
result4 = re.search("\d+",result3) # 문자열 속 숫자부분 찾기 
search_cnt = int(result4.group()) # 찾은 문자열 반환, 정수로 변환 

# 사용자가 요청한 개수와 실제 개수 맞추기 ~ 예외처리, 실제댓글이 100개인데 사용자가 500개를 원할 때. 
if cnt > search_cnt :
    cnt = search_cnt

page_cnt = math.ceil(cnt / 20)

print("전체 검색 결과 건수 :",search_cnt,"건")
print("실제 최종 출력 건수",cnt)
print("실체 출력될 최종 페이지수" , page_cnt)



전체 검색 결과 건수 : 505 건
실제 최종 출력 건수 30
실체 출력될 최종 페이지수 2


<>:12: SyntaxWarning: invalid escape sequence '\d'
<>:12: SyntaxWarning: invalid escape sequence '\d'
C:\Users\itwill\AppData\Local\Temp\ipykernel_7452\3273116814.py:12: SyntaxWarning: invalid escape sequence '\d'
  result4 = re.search("\d+",result3) # 문자열 속 숫자부분 찾기


In [5]:
# Step 5. 사용자가 요청한 건수가 많을 경우 리뷰 더보기 버튼을 클릭합니다
# 최초 10건 수집후 댓글 더보기 버튼 클릭
# 아래 버튼을 눌러 첫 화면에 총 20건의 댓글이 나오게 만듦
#driver.find_element(By.XPATH,'//*[@id="cbox_module"]/div[2]/div[9]/a/span/span/span[1]').click()
driver.find_element(By.XPATH,'//*[@id="cbox_module"]/div[2]/div[9]/a').click()

#//*[@id="cbox_module"]/div[2]/div[9]/a
time.sleep(3)



In [6]:
#Step 6. 리뷰와 점수 등 내용 수집
#집한 데이터를 저장할 빈 리스트를 만드는 부분 ~ 각 리스트는 나중에 열column이 된다. 
# 작성자ID	리뷰내용	작성일자	공감횟수	비공감횟수
writer_id2=[] # 리뷰 작성자 ID
review2=[] # 리뷰 내용
write_date2=[] # 리뷰 작성 일자
gogam_0=[] # 공감 횟수
gogam_1=[] # 비공감 횟수
count = 0

# 더보기 버튼 반복 클릭 
for a in range(1,page_cnt+1) :

    if a == page_cnt :
        break
    else :
        driver.find_element(By.XPATH,'//*[@id="cbox_module"]/div[2]/div[9]/a').click()
        time.sleep(3)
        print("%s페이지 이동 완료============================" %a)
        time.sleep(random.randrange(1,3)) # 3-8 초 사이에 랜덤으로 시간 선택

print('이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~')



1페이지 이동 완료============================
이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~


In [7]:
# 1. 전체 영역 찾기
# 2. 반복되는 요소들을 리스트로 가져오기
# 3. 반복문으로 하나씩 처리

# 댓글 영역
#  └ 댓글1
#  └ 댓글2
#  └ 댓글3
#  └ 댓글4

# div (댓글 전체 영역)
#    ul
#       li (댓글1)
#       li (댓글2)
#       li (댓글3)
#       li (댓글4)

# 🔑 크롤링에서 가장 중요한 사고법
# 1. 무엇이 반복되는가? ~ 반복되는 구조 찾기
# 2. 그 반복되는 태그는 무엇인가? ~ 그 태그를 리스트로 가져오기
# 3. find_all로 리스트를 만든다 
# 4. 반복문으로 하나씩 처리

# 반복문 구조 (가장 기본 형태)
# 리스트 = 여러 요소 가져오기

# for 변수 in 리스트:
#     데이터 추출

# 크롤링 반복문 기본 템플릿
# items = soup.find_all("반복되는태그")

# for item in items:

#     data1 = item.find(...)
#     data2 = item.find(...)
#     data3 = item.find(...)

#     리스트.append(data1)

# Selenium에서도 같은 구조   
# items = driver.find_elements(By.CSS_SELECTOR, ".product")

# for item in items:

#     title = item.find_element(...)
#     price = item.find_element(...)

In [8]:
# 코드에서 실제로 하는 일
# ① 댓글 전체 영역 찾기
# reple_result = soup.find('div', class_='u_cbox_content_wrap').find('ul')

# HTML 구조

# div class="u_cbox_content_wrap"
#    ul
#       li
#       li
#       li

# 그래서

# 댓글 전체 영역 = ul

# 을 먼저 찾습니다.

# ② 댓글들을 리스트로 가져오기
# slist = reple_result.find_all('li')

# 이 코드의 의미

# li 태그 전부 가져와라

# 결과는

# [
# 댓글1 li,
# 댓글2 li,
# 댓글3 li,
# 댓글4 li,
# 댓글5 li
# ]

# 이렇게 리스트가 됩니다.

# 그래서 변수 이름도

# slist

# 라고 만든 것입니다.

In [9]:
#txt 파일에 저장하기 위해 파일 open하기
# 'a'는 append 모드, 즉 이어쓰기 모드. 기존 내용이 있으면 뒤에 추가, 없으면 새로 생성
f = open(ff_name, 'a',encoding='UTF-8')

# 다시 html 가져오기 
# 더보기 버튼을 여러 번 누른 후에는 HTML 구조가 바뀌었기 때문에 
# 처음 HTML이 아니라
# 지금 화면에 실제로 펼쳐진 댓글 상태의 HTML을 다시 읽어와야 해
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

reple_result = soup.find('div', class_='u_cbox_content_wrap').find('ul')# .fin
slist = reple_result.find_all('li')

for li in slist:
    count += 1
    print("\n")
    print("총 %s건 중 %s번째 댓글 수집 중입니다 =========" %(cnt,count))

    writer_id = li.find('span', class_='u_cbox_nick').get_text()
    print("1.작성자ID:", writer_id)
    f.write("\n")
    f.write("총 %s 건 중 %s 번째 리뷰 데이터를 수집합니다====" %(cnt,count) + "\n")
    f.write("1.작성자ID:"+writer_id + "\n")
    writer_id2.append(writer_id)

    try :
        review = li.find('span', class_='u_cbox_contents').get_text()
    except AttributeError :
        review='작성자에 의해 삭제된 댓글입니다' #태그가 없어서 오류가 나면 삭제된 댓글이라고 간주 
        print("2.리뷰 :",review)
    else :
        print("2.리뷰:",review)
        f.write("2.리뷰:" + review + "\n")
    review2.append(review)

    write_date = li.find('span',class_='u_cbox_date').get_text()
    print('3.작성일자:',write_date)
    f.write("3.작성일자:" + write_date + "\n")
    write_date2.append(write_date)

    gogam = li.find('div', class_='u_cbox_recomm_set').find_all('em')

    try :
        g_gogam = gogam[0].text
        print('4.공감:',g_gogam)
    except IndexError :
        g_gogam = '0'
        print('4.공감 :',g_gogam)
    f.write("4.공감:" + g_gogam + "\n")
    gogam_0.append(g_gogam)

    gogam = li.find('div', class_='u_cbox_recomm_set').find_all('em')

    try :
        b_gogam = gogam[1].text
        print('5.비공감:',b_gogam)
    except IndexError :
        b_gogam = '0'
        print('5.비공감 :',b_gogam)
    f.write("5.비공감:" + b_gogam + "\n")
    gogam_1.append(b_gogam)

    time.sleep(0.2)

    if count == cnt :
        break

f.close()





총 30건 중 1번째 댓글 수집 중입니다 =========
1.작성자ID: ehdd****
2.리뷰: 어떻게든 한국미세먼지 탓도있다고 보도하려고 애쓰네ㅋㅋ
3.작성일자: 2019.01.15. 16:10
4.공감: 2018
5.비공감: 2018


총 30건 중 2번째 댓글 수집 중입니다 =========
1.작성자ID: siho****
2.리뷰: 진짜 쓰레기 보다 못한 기사다 그래서 어쩌 자는거냐? 누가 심한거 모르냐? 정상 적인 언론이라면 지금에 미세먼지 원인이 뭐고 앞으로 나가야할 정부 정책에는 뭐가 있고 뭔가 대책과 무능한 정부에 질책이나 문제점을 쓰는게 맞을듯 싶은데 누가 미세먼지 나쁘다는것에 부정하는 사람있냐? 국민세금이 아깝다
3.작성일자: 2019.01.15. 16:26
4.공감: 725
5.비공감: 725


총 30건 중 3번째 댓글 수집 중입니다 =========
1.작성자ID: reds****
2.리뷰: 충남공장 사진은 왜 올리는거지
3.작성일자: 2019.01.15. 16:38
4.공감: 519
5.비공감: 519


총 30건 중 4번째 댓글 수집 중입니다 =========
1.작성자ID: daup****
2.리뷰 : 작성자에 의해 삭제된 댓글입니다
3.작성일자: 2019.12.05. 22:23
4.공감 : 0
5.비공감 : 0


총 30건 중 5번째 댓글 수집 중입니다 =========
1.작성자ID: samk****
2.리뷰: 애쓴다ㅋㅋㅋㅋㅋㅋ
3.작성일자: 2019.01.15. 16:55
4.공감: 137
5.비공감: 137


총 30건 중 6번째 댓글 수집 중입니다 =========
1.작성자ID: mark****
2.리뷰: 기자님. 잘 모르는건 대충 짐작으로 글 적지 마세요. 중간 충남 공장 사진에 오염물질을 뿜어대는 공장이라고 글을 다셨는데, 저거 거의 다 수증기입니다. TEMS 때문에 실시간 감시되서 오염물질 뿜어대질 못해요. 제대로 조사 안한 내용이나 모르는건 상상력으로 기사 적지 마세요. 그러니 잘 못된 기사가 나

In [10]:
#Step 7. xlsx 형태와 csv 형태로 저장하기
news_reple = pd.DataFrame()
news_reple['작성자ID']=pd.Series(writer_id2)
news_reple['리뷰내용']=pd.Series(review2)
news_reple['작성일자']=pd.Series(write_date2)
news_reple['공감횟수']=pd.Series(gogam_0)
news_reple['비공감횟수']=pd.Series(gogam_1)

# csv 형태로 저장하기
news_reple.to_csv(fc_name,encoding="utf-8-sig",index=True)

# 엑셀 형태로 저장하기
news_reple.to_excel(fx_name ,index=False , engine='openpyxl')




In [11]:
# Step 8. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time
print("\n")
print("=" *120)
print("1.요청된 총 %s 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 %s 건입니다" %(cnt,count))
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xlsx 파일명 : %s " %fx_name)
print("=" *120)
driver.close( )


print("txt 존재 여부:", os.path.exists(ff_name), os.path.getsize(ff_name))
print("csv 존재 여부:", os.path.exists(fc_name), os.path.getsize(fc_name))
print("xlsx 존재 여부:", os.path.exists(fx_name), os.path.getsize(fx_name))




1.요청된 총 30 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 30 건입니다
2.총 소요시간은 20.4 초 입니다 
3.파일 저장 완료: txt 파일명 : c:\py_temp\2026-03-11-14-28-54-뉴스기사댓글\2026-03-11-14-28-54-뉴스기사댓글.txt 
4.파일 저장 완료: csv 파일명 : c:\py_temp\2026-03-11-14-28-54-뉴스기사댓글\2026-03-11-14-28-54-뉴스기사댓글.csv 
5.파일 저장 완료: xlsx 파일명 : c:\py_temp\2026-03-11-14-28-54-뉴스기사댓글\2026-03-11-14-28-54-뉴스기사댓글.xlsx 
txt 존재 여부: True 8885
csv 존재 여부: True 5740
xlsx 존재 여부: True 8264
